# 🧬 Project 5 — Semantic Correspondence (Official Version)
**Team:** Johnprice Osagie · Mario Lapadula · Giorgia Pugliese · Riccardo Bellanca

Pipeline avanzata con analisi multi-backbone, PEFT, raffinamento spaziale adattivo e test di generalizzazione.

## 📦 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone -b main https://github.com/Johnnyprice29/Project_AML.git /content/Project_AML
%cd /content/Project_AML
!git fetch origin && git reset --hard origin/main
!pip install -r requirements.txt -q
!pip install gradio -q
!python dataloaders/download_spair.py --root ./data

import os
from utils.demo_utils import launch_stage_demo, launch_comparison_demo

## 🔍 1. Stage 1: Multi-Backbone Analysis (Baseline)
In questa fase eseguiamo l'estrazione sistematica dei risultati per i diversi backbone suggeriti dalla traccia, valutando la robustezza con le soglie 0.2, 0.1 e 0.05.

### 🦖 1.1 Confronto DINOv2 vs DINOv3 vs SAM
Eseguiamo la valutazione 'zero-shot' per identificare il miglior punto di partenza.

In [ ]:
# DINOv2 Baseline
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov2_vitb14 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov2.txt

# DINOv3 Baseline
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone dinov3_vitb14 --results_file /content/drive/MyDrive/AML/Results/baseline_dinov3.txt

# SAM Baseline
!python evaluate.py --dataset_root ./data/SPair-71k --baseline_only --backbone sam_vitb --results_file /content/drive/MyDrive/AML/Results/baseline_sam.txt

### 🔦 1.2 Layer Explorer (Chosen Backbone: DINOv2)

In [ ]:
launch_stage_demo('DINOv2 Layer Explorer', show_layer_slider=True)

## 🚀 2. Stage 2: Fine-Tuning Efficiente (PEFT)
Ottimizziamo il modello scelto tramite LoRA e BitFit. La cella è idempotente e rileva se i modelli sono già su Drive.

In [ ]:
DRIVE_CKPTS = '/content/drive/MyDrive/AML/Checkpoints'
if not os.path.exists(f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'):
    !python train.py --peft_type lora --dataset_root ./data/SPair-71k --epochs 5 --exp_name lora_only --output_dir ./checkpoints/lora_only --backup_dir {DRIVE_CKPTS}/lora_only
else: print('[INFO] Checkpoint LoRA trovato. Salto training.')

## 🎯 3. Stage 3: Raffinamento e Risultati Finali
Valutazione LoRA + Adaptive Window (0.2, 0.1, 0.05).

In [ ]:
DRIVE_RESULTS = '/content/drive/MyDrive/AML/Results'
CKPT_LORA = f'{DRIVE_CKPTS}/lora_only/lora_only_best.pth'
!python evaluate.py --dataset_root ./data/SPair-71k --checkpoint {CKPT_LORA} --results_file {DRIVE_RESULTS}/final_results_extended.txt
launch_stage_demo('LoRA + Adaptive Window', ckpt_name='lora_only')

## 📏 4. Stage 4: Robustezza Geometrica
Test di resilienza alle trasformazioni dello spazio.

In [ ]:
launch_comparison_demo(ckpt_name='lora_only')

## 🌍 5. Stage 5: Generalizzazione (New Domains)
Valutazione su dataset esterni.

In [ ]:
print('Pronto per test su PF-Pascal o nuove immagini caricabili manualmente...')